# 0. Imports

In [ ]:
%%capture
%pip install easyocr pytesseract codecarbon

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, AutoModelForImageTextToText, LightOnOcrForConditionalGeneration, LightOnOcrProcessor
import numpy as np
import cv2
import datetime as dt
import os, json, re, time
from bs4 import BeautifulSoup
import easyocr
import pytesseract
from codecarbon import EmissionsTracker

# 1. Parameters

In [ ]:
avail_models = ["easyocr", "pytesseract", "google/gemma-3-4b-it", "lightonai/LightOnOCR-2-1B", "zai-org/GLM-OCR", "PaddlePaddle/PaddleOCR-VL-1.6", "Qwen/Qwen2.5-VL-7B-Instruct"]
avail_datasets = ["ICDAR03/apanar", "ICDAR03/lfsosa", "ICDAR03/ryoungt1", "ICDAR03/ryoungt2", "ICDAR13/train", "ICDAR13/test", "kahua-ml/flattened-nameplate", "SVT/train", "icare/train"]

lighton= "lightonai/LightOnOCR-2-1B"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32 if device == "cpu" else torch.bfloat16

BATCH_SIZE = 50 if device == "cuda" else 10
HF_TOKEN = os.environ.get("HF_TOKEN", None)
MARGIN = 10 # pixels to add to bounding boxes when cropping
NBR_TOKENS = 1024
USE_BBOX = True
IS_LLM = None
CONF_THRESHOLD = 60

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    is_colab = True
except ImportError:
    print("Google Colab environment not detected.")
    is_colab = False

If we are on Colab, we need to update pytesseract to a version >= 5.0

PS: It asks you to press 'ENTER' to continue

In [ ]:
if is_colab:
    !sudo add-apt-repository ppa:alex-p/tesseract-ocr-devel
    !sudo apt-get update
    !sudo apt-get install tesseract-ocr
    # Get the language packages from version 4
    os.environ["TESSDATA_PREFIX"] = "/usr/share/tesseract-ocr/4.00/tessdata"
    print(pytesseract.get_tesseract_version())

In [ ]:
def define_global_paths():
    global DATASET_PATH, SAVE_PATH, RESULTS_PATH
    DATASET_PATH = "resources/datasets/" if not is_colab else "/content/drive/MyDrive/Thesis/resources/datasets/"
    SAVE_PATH = "output/predictions/" if not is_colab else "/content/drive/MyDrive/Thesis/resources/predictions/"
    RESULTS_PATH = "output/results/" if not is_colab else "/content/drive/MyDrive/Thesis/resources/results/"

define_global_paths()

# 2. Model preparation
Model download and initialization

In [ ]:
class TesseractOCR:
    """
    A wrapper class for Tesseract OCR using pytesseract.
    It allows for easy configuration of language, page segmentation mode (PSM).
    """
    def __init__(self, lang="eng", psm=6):
        """
        Initialize the TesseractOCR instance.

        Args:
            lang (str): The language to use for OCR. Default is "eng" (English).
            psm (int): The page segmentation mode. Default is 6 (treat the image as a single uniform block of text). If USE_BBOX is True, psm will be set to 7 (assume a single line of text).
        """
        self.lang = lang
        self.psm = psm if USE_BBOX else 6
        self._config = self._build_config()

    def _build_config(self):
        config = f"--psm {self.psm}"
        return config

    def read(self, image):
        return pytesseract.image_to_string(image, lang=self.lang, config=self._config)

    def read_data(self, image):
        return pytesseract.image_to_data(image, lang=self.lang, config=self._config, output_type=pytesseract.Output.DATAFRAME)

class Prediction:
    """
    %%Not Used
    A class to represent a prediction made by an OCR model.
    It contains the predicted word, its confidence score, and the processing time taken to make the prediction.
    """
    def __init__(self, word: str, confidence: float, processing_time: float):
        self.word = word
        self.confidence = confidence

    def __repr__(self):
        return f"Prediction(word={self.word}, confidence={self.confidence}, processing_time={self.processing_time})"

In [ ]:
def init_model(model_name:str):
    global IS_LLM
    IS_LLM = model_name not in ["easyocr", "pytesseract"]
    if IS_LLM:
        global USE_BBOX
        USE_BBOX = False
        model, processor = try_load_llm_model(model_name)
        reader = None
    elif model_name == "easyocr":
        USE_BBOX = True
        reader = easyocr.Reader(['en', 'fr'], gpu=torch.cuda.is_available())
        model, processor = None, None
    elif model_name == "pytesseract":
        USE_BBOX = True
        reader = TesseractOCR(lang="eng", psm=8)
        model, processor = None, None
    else:
        raise ValueError(f"Model {model_name} not supported. Available models: {avail_models}")
    return model, processor, reader

def set_instructions(model_name:str):
    global IS_LLM
    if not IS_LLM:
        return []
    match model_name:
        case "LightOnOCR-2-1B":
            return []
        case _:
            instructions = [
                "You are an OCR engine. You do not chat, explain, or comment.",
                "Locate all text in the image and transcribe it exactly as it appears.",
                "Output ONLY the raw transcribed text. Nothing else.",
                "Do NOT add any introduction (e.g. 'Here is the text...').",
                "Do NOT add any conclusion, offer, or question (e.g. 'Let me know if...').",
                "Do NOT use Markdown formatting: no bullet points, no bold (**), no headers (#), no asterisks.",
                "Do NOT interpret, label, or reformat the content — just transcribe what is visually present, line by line.",
                "Your entire response must be the transcription and nothing else.",
            ]
            return instructions

def try_load_llm_model(model_name):
    model_suffix = model_name.split("/")[-1]
    if os.path.exists(f"Models/{model_suffix}"):
        print(f"Loading model {model_name} from Models/{model_suffix}")
        if model_name == lighton:
            model = LightOnOcrForConditionalGeneration.from_pretrained(lighton, dtype=dtype).to(device)
            processor = LightOnOcrProcessor.from_pretrained(lighton)
        elif model_name == "PaddlePaddle/PaddleOCR-VL-1.6":
            processor = AutoProcessor.from_pretrained(f"Models/{model_suffix}", use_auth_token=HF_TOKEN)
            model = AutoModelForImageTextToText.from_pretrained(f"Models/{model_suffix}").to(device)
        else:
            processor = AutoProcessor.from_pretrained(f"Models/{model_suffix}", use_auth_token=HF_TOKEN)
            model = AutoModelForMultimodalLM.from_pretrained(f"Models/{model_suffix}").to(device)
        return model, processor
    else:
        print(f"Downloading model {model_name}, and saving in Models/{model_suffix}")
        if model_name == lighton:
            model = LightOnOcrForConditionalGeneration.from_pretrained(lighton, dtype=dtype).to(device)
            processor = LightOnOcrProcessor.from_pretrained(lighton)
        elif model_name == "PaddlePaddle/PaddleOCR-VL-1.6":
            processor = AutoProcessor.from_pretrained(model_name, use_auth_token=HF_TOKEN)
            model = AutoModelForImageTextToText.from_pretrained(model_name).to(device)
        else:
            processor = AutoProcessor.from_pretrained(model_name, use_auth_token=HF_TOKEN, trust_remote_code=True)
            model = AutoModelForMultimodalLM.from_pretrained(model_name, trust_remote_code=True).to(device)
        processor.save_pretrained(f"Models/{model_suffix}")
        model.save_pretrained(f"Models/{model_suffix}")
        return model, processor

# 3. Inference

## 3.1 Definitions

In [ ]:
def natural_sort_key(value):
    parts = re.split(r'(\d+)', value)
    return [int(part) if part.isdigit() else part.lower() for part in parts]

def save_predictions_checkpoint(path, data):
    # Make sure the directory exists
    os.makedirs(os.path.dirname(path), exist_ok=True)
    temp_path = f"{path}.tmp"
    with open(temp_path, "w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2, ensure_ascii=False)
    os.replace(temp_path, path)

def try_resume_from_checkpoint(checkpoint_path):
    if os.path.exists(checkpoint_path) and os.path.getsize(checkpoint_path) > 0:
        with open(checkpoint_path, "r", encoding="utf-8") as handle:
            try:
                data = json.load(handle)
                print(f"Resuming from checkpoint: {checkpoint_path} with {len(data)} entries")
                return data
            except json.JSONDecodeError:
                print(f"Checkpoint file {checkpoint_path} is corrupted. Starting fresh.")
                return {}
    return {}

def get_resized_roi(image, coords):
    """Get a resized region of interest from the image based on coordinates."""
    if coords is None:
        return image

    if len(coords) == 0:
        return image
    elif len(coords) == 4:
        x1, y1, x2, y2 = map(int, coords)
        x1 = max(0, x1 - MARGIN)
        y1 = max(0, y1 - MARGIN)
        x2 = min(image.shape[1], x2 + MARGIN)
        y2 = min(image.shape[0], y2 + MARGIN)
        roi = image[y1:y2, x1:x2]
    elif len(coords) == 8:
        x1, y1, x2, y2, x3, y3, x4, y4 = map(int, coords)
        x1 = max(0, x1 - MARGIN)
        y1 = max(0, y1 - MARGIN)
        x3 = min(image.shape[1], x3 + MARGIN)
        y3 = min(image.shape[0], y3 + MARGIN)
        roi = image[y1:y3, x1:x3]
    elif len(coords) > 8 and len(coords) % 2 == 0:
        x_s = [coords[i] for i in range(0, len(coords), 2)]
        y_s = [coords[i] for i in range(1, len(coords), 2)]
        x1, y1 = max(0, min(x_s) - MARGIN), max(0, min(y_s) - MARGIN)
        x2, y2 = min(image.shape[1], max(x_s) + MARGIN), min(image.shape[0], max(y_s) + MARGIN)
        roi = image[y1:y2, x1:x2]
    else:
        return None
    return roi

def extract_content(string, model_name:str):
    """Short function to extract the content from the string based on the model name."""
    suffix = model_name.split("/")[-1]
    match suffix:
        case "LightOnOCR-2-1B":
            results = light_on_ocr_to_word_list(string)
            return results
        case "gemma-3-4b-it" | "PaddleOCR-VL-1.6" | "Qwen2.5-VL-7B-Instruct" | "GLM-OCR":
            # For gemma-3-4b-it, we need to extract the JSON content
            return gemma_3_to_word_list(string)
        case "NuExtract3":
            return [string]
        case _:
            return [string]

def gemma_3_to_word_list(text: str) -> list[str]:
    words = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue  # on ignore les lignes vides
        words.extend(line.split())  # découpe la ligne en mots sur les espaces
    return words

def light_on_ocr_to_word_list(text: str) -> list[str]:
    # We remove all the text that is after the "Note:" line, as it is not part of the transcription
    img_index = text.lower().find("![image]")
    if img_index != -1:
        text = text[:img_index]
    note_index = text.lower().find("Note:")
    if note_index != -1:
        text = text[:note_index]

    soup = BeautifulSoup(text, "html.parser")
    text_no_html = soup.get_text(separator=" ")

    text_clean = re.sub(r'\|', ' ', text_no_html)
    text_clean = re.sub(r'\$.*?\$', '', text_clean)  # Remove math mode
    text_clean = re.sub(r'(?im)^\s*Note:\s*.*$', '', text_clean)
    text_clean = re.sub(r'^[-:=]{3,}$', '', text_clean, flags=re.MULTILINE)
    text_clean = re.sub(r'#{1,6}\s*', '', text_clean)
    text_clean = re.sub(r'\*{1,2}([^*]+)\*{1,2}', r'\1', text_clean)
    text_clean = re.sub(r'^\s*[-*+]\s+', '', text_clean, flags=re.MULTILINE)

    words = text_clean.strip().split()

    return words

def extract_content_from_bbox_results(results, model_name:str):
    """
    Extract the content from the results based on the model name.
    For easyocr, results is a list of tuples (bbox, text, prob).
    For pytesseract, results is a dictionary with keys 'text' and 'conf'."""
    if model_name == "easyocr":
        conf = 0.0
        word = ""
        for (bbox, text, prob) in results:
            if prob > conf:
                conf = float(prob)
                word = str(text).strip()
        return word, round(conf, 4)
    else:
        # For pytesseract
        conf = 0.0
        word = ""
        for i in range(len(results['text'])):
            text = results['text'][i]
            prob = results['conf'][i]
            if prob > conf:
                conf = float(prob)
                word = str(text).strip()
        return word, round(conf, 4)

def easyocr_pytesseract_to_word_list(results):
    words = []
    if model_name == "easyocr":
        for (bbox, text, prob) in results:
            conf = round(float(prob)*100, 4)
            text = str(text).strip()
            if conf > CONF_THRESHOLD:
                print(f"Text: '{text}', Confidence: {conf}")
                # If we have one, two and less than 20% of the length of the text that are whitespace, we do a split on the text and add each word separately.
                # Else, we add the whole text as one word.
                if text.count(" ") <= 2:
                    if text.count(" ") <= len(text) * 0.2:
                        for word in text.split():
                            words.append((word, conf))
                    else:
                        # Single word with some whitespace
                        text = text.replace(" ", "")
                        words.append((text, conf))
                else:
                    if text.count(" ") <= len(text) * 0.2:
                        # Several complete words, we split them and add each word separately.
                        for word in text.split():
                            words.append((word, conf))
                    else:
                        # Many short words, we add each word separately.
                        for word in text.split():
                            words.append((word, conf))
    else:
        # For pytesseract
        for i in range(len(results['text'])):
            text = results['text'][i]
            prob = results['conf'][i]
            conf = round(float(prob), 4)
            text = str(text).strip()
            if conf > CONF_THRESHOLD:
                words.append((text, conf))
    return words

def save_predictions_to_json(predictions_dict, dataset:str, model_name:str, use_bbox:bool):
    """Save the predictions dictionary to a JSON file."""
    mainset_name, subset_name = dataset.split("/")
    company_name, model_nme = model_name.split("/") if '/' in model_name else (None, model_name)
    # Make sure the directory exists
    save_dir = os.path.join(SAVE_PATH, mainset_name)
    os.makedirs(save_dir, exist_ok=True)
    save_file_path = os.path.join(save_dir, f"{subset_name}_{model_nme}{'_bb' if use_bbox else ''}.json")
    with open(save_file_path, "w", encoding="utf-8") as f:
        json.dump(predictions_dict, f, indent=2, ensure_ascii=False)
    print(f"Predictions saved to {save_file_path}")


In [ ]:
def benchmark(model_name:str = "easyocr", datasets:list[str] = ["ICDAR03/apanar"], use_bbox:bool = True, batch_size:int = BATCH_SIZE, nbr_tokens:int = NBR_TOKENS, use_checkpoint:bool = True, record_emissions:bool = True):
    # Model initialization
    model, processor, reader = init_model(model_name)
    instructions = set_instructions(model_name)

    # Dataset processing
    for dataset in datasets:
        # Check if dataset exists
        if not os.path.exists(f"{DATASET_PATH}/{dataset}"):
            print(f"Dataset {dataset} not found in {DATASET_PATH}. Skipping.")
            continue

        print(f"Processing dataset {dataset}")
        # The predictions dictionary will store the results for each image in the dataset. The key is the image name (without extension), and the value is a dictionnary with two keys: "predictions" and "time". "predictions" is a list of tuples (word, confidence) and "time" is the processing time in seconds for that image.
        predictions_dict = {}
        batch_dict = {}

        try:
            _, model_nme = model_name.split("/")
        except:
            model_nme = model_name
        main, sub = dataset.split("/")

        # Defining the different paths.
        if not is_colab:
            save_path = f"{SAVE_PATH}{main}/{sub}_{model_nme}{'_bb' if use_bbox else ''}.json"
            checkpoint_path = f"resources/checkpoints/{dataset.replace('/', '_')}_{model_nme}{'_bb' if use_bbox else ''}.json"
            emissions_dir = "resources/emissions"
        else:
            save_path = f"/content/drive/MyDrive/Thesis/resources/predictions/{main}/{sub}_{model_nme}{'_bb' if use_bbox else ''}.json"
            checkpoint_path = f"/content/drive/MyDrive/Thesis/resources/checkpoints/{dataset.replace('/', '_')}_{model_nme}{'_bb' if use_bbox else ''}.json"
            emissions_dir = "/content/drive/MyDrive/Thesis/resources/emissions"

        # Load metadata if bounding boxes are used
        if use_bbox:
            metadata_file = json.load(open(f"{DATASET_PATH}{dataset}/data.json", encoding="utf-8"))
            word_count = sum(len(coords_list) for coords_list in metadata_file.values())
            print(f"Metadata loaded for {word_count} words in {len(metadata_file)} images.")

        # Load checkpoint if available
        if use_checkpoint:
            try:
                _, model_nme = model_name.split("/")
            except:
                model_nme = model_name

            predictions_dict = try_resume_from_checkpoint(checkpoint_path)

            # If a final prediction file already exists, skip the dataset entirely
            main, sub = dataset.split("/")

            if os.path.exists(save_path):
                print(f"Final predictions already exist at {save_path}. Skipping dataset {dataset}.")
                continue

        # Load all images in the dataset
        image_files = sorted(
            [file for file in os.listdir(f"{DATASET_PATH}/{dataset}") if file.lower().endswith((".jpg", ".jpeg", ".png"))],
            key=lambda file: natural_sort_key(os.path.splitext(file)[0])
        )
        if len(image_files) == 0:
            print(f"No images found in dataset {dataset}. Skipping.")
            continue

        # Start the CodeCarbon tracker
        if record_emissions:
            os.makedirs(emissions_dir, exist_ok=True)
            tracker = EmissionsTracker(project_name=f"{dataset}_{model_name}{'_bb' if use_bbox else ''}", output_dir=emissions_dir, save_to_file=True, log_level="error")
            tracker.start()

        start_time = dt.datetime.now()
        last_checkpoint_time = start_time

        # Process each batch of images
        for batch_start in range(0, len(image_files), batch_size):
            batch_files = image_files[batch_start:batch_start + batch_size]
            batch_names = [os.path.splitext(file)[0] for file in batch_files]

            if not batch_names:
                continue    # Skipping empty batches

            # Treat each image in the batch
            for image_file in batch_files:
                image_path = f"{DATASET_PATH}/{dataset}/{image_file}"
                image_key = os.path.splitext(image_file)[0]
                image = cv2.imread(image_path)

                if image_key in predictions_dict:
                    continue

                if image is None:
                    print(f"Failed to load image {image_path}. Skipping.")
                    continue

                # Processing logic based on model type
                if IS_LLM:
                    # Prepare the input for the LLM model
                    messages = [{"role": "user","content": [{"type": "image", "image": image},{"type": "text", "text": "".join(instructions)},]}]

                    # Start the execution timer for LLM processing
                    exec_start_time = dt.datetime.now() # Mostly to track individual image processing time, to make cactus plot.

                    inputs = processor.apply_chat_template(
                        messages,
                        add_generation_prompt=True,
                        tokenize=True,
                        return_dict=True,
                        return_tensors="pt",
                    ).to(model.device)

                    outputs = model.generate(**inputs, max_new_tokens=nbr_tokens)
                    generated_ids = outputs[0, inputs["input_ids"].shape[1]:]
                    result = processor.decode(generated_ids, skip_special_tokens=True)

                    exec_end_time = dt.datetime.now()
                    process_time = exec_end_time - exec_start_time
                    process_time = round(process_time.total_seconds(), 4)
                    list_of_words = extract_content(result, model_name)
                    list_of_words_with_conf = [(word, None) for word in list_of_words]

                    predictions_dict[image_key] = {"predictions": list_of_words_with_conf, "time": process_time}

                else:
                    # print(f"Running OCR model {model_name} on dataset {dataset}...")
                    list_of_words_with_conf = []
                    glob_time = 0.0
                    if use_bbox:
                        #print("Bounding box usage is enabled.")
                        # We only treat the images that have bounding boxes in the metadata file
                        for gt_key, coords_list in metadata_file.get(image_key, {}).items():
                            for coords in coords_list:
                                roi = get_resized_roi(image, coords)
                                if roi is None:
                                    raise ValueError(f"Invalid coordinates {coords} for image {image_file}. Could not extract ROI.")

                                exec_start_time = dt.datetime.now()
                                if model_name == "easyocr":
                                    result = reader.readtext(roi, detail=1, paragraph=False)
                                    (bbox, text, prob) = result[0] if result else (None, "", 0)
                                else:
                                    reader = TesseractOCR(lang="eng", psm=8)
                                    result = reader.read_data(roi)  # For pytesseract

                                exec_end_time = dt.datetime.now()
                                process_time = exec_end_time - exec_start_time
                                process_time = round(process_time.total_seconds(), 4)
                                # We add the processing time to the global time for the image
                                glob_time += process_time

                                word, conf = extract_content_from_bbox_results(result, model_name)

                                list_of_words_with_conf.append((word, conf))
                        glob_time = round(glob_time, 4)

                    else:
                        exec_start_time = dt.datetime.now()
                        if model_name == "easyocr":
                            result = reader.readtext(image, detail=1, paragraph=False)
                        else:
                            reader = TesseractOCR(lang="eng", psm=6)
                            result = reader.read_data(image)  # For pytesseract
                        exec_end_time = dt.datetime.now()
                        process_time = exec_end_time - exec_start_time
                        glob_time = round(process_time.total_seconds(), 4)

                        results_list = easyocr_pytesseract_to_word_list(result)
                        print(f"Results for image {image_key}: {results_list}")

                        list_of_words_with_conf = [(word, conf) for word, conf in results_list]

                    predictions_dict[image_key] = {"predictions": list_of_words_with_conf, "time": glob_time}

            batch_step_time = dt.datetime.now()
            elapsed_time = batch_step_time - last_checkpoint_time
            minutes, seconds = divmod(elapsed_time.total_seconds(), 60)
            seconds, micro_seconds = divmod(seconds, 1)
            last_checkpoint_time = batch_step_time

            save_predictions_checkpoint(checkpoint_path, predictions_dict)
            #print(f"Checkpoint saved after batch [{batch_names[0]} to {batch_names[-1]}]: {len(batch_files)} images in {int(minutes)} minutes and {int(seconds)} seconds.")

            # We had the time took to process the batch.
            batch_number = batch_start // batch_size + 1
            b_key = f"Batch #{batch_number}"
            batch_dict[b_key] = {
                "time": f"{int(minutes)}min_{int(seconds)}s_{int(micro_seconds * 1000)}ms",
                "size": len(batch_files)
            }

        tracker.stop()

        predictions_dict.update({"batch_info": batch_dict})
        # Save the final predictions to a JSON file
        save_predictions_to_json(predictions_dict, dataset, model_name, use_bbox)


## 3.2 Inference

### 3.2.1 Global Inference

In [ ]:
datasets_to_exclude = ["ICDAR15/train", "ICDAR15/test"]
datasets_to_run = [dataset for dataset in avail_datasets if dataset not in datasets_to_exclude]
model_name = "easyocr"  # Change this to the desired model name
#avail_datasets=['ICDAR03/apanar']

try:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
except:
    pass

benchmark(model_name=model_name,
          use_bbox=False,
          batch_size=BATCH_SIZE,
          nbr_tokens=NBR_TOKENS,
          use_checkpoint=False,
          record_emissions=False,
          )

print(f"Pic VRAM allouée: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print(f"Pic VRAM réservée: {torch.cuda.max_memory_reserved() / 1024**3:.2f} GB")

### 3.2.2 Unique image inference

In [ ]:
#TODO: Add a function to infer on a single image, given the model name and the image path.

# 4. Metrics

In [ ]:
import Levenshtein as lev
import pandas as pd
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as tck
from scipy.optimize import linear_sum_assignment
import seaborn as sns
from difflib import SequenceMatcher as sm
import numpy as np

## 4.1 Creation

### 4.1.1 Definitions

In [ ]:
def clean_float(string):
    # When int are read, they were casted to float ("23": "23.0",)
    try:
        if "." in string and float(string) == int(float(string)):
            return str(int(float(string)))
    except ValueError:
        pass
    return string

def get_words_lists(dataset:str):
    main, sub = dataset.split("/")
    with open(f"{DATASET_PATH}{main}/{sub}/data.json", 'r', encoding="utf-8") as f:
        data = json.load(f)
        
    gt_words = {}
    for img_name, words in data.items():
        word_list = []
        if type(words) is not dict:
            # Handle the case where we only have a list of words instead of a dictionary (i-care dataset)
            for w in words:
                cleaned_word = clean_float(w)
                word_list.append(cleaned_word)
        else:
            for word in data[img_name].keys():
                cleaned_word = clean_float(word)
                word_list.append(cleaned_word)
        gt_words[img_name] = word_list
    return gt_words

def hungarian_align(gt_tokens:list, pred_tokens:list):
    """
    Align GT tokens with predicted tokens using Hungarian matching on
    Levenshtein distance.

    If len(gt_tokens) > len(pred_tokens), missing predictions are modeled
    as empty strings ("") so every GT token gets matched.
    Extra predicted tokens are compared to the empty string and will have a distance equal to their length.
    
    Then we shrink the cost matrix to only include the GT tokens and their matched predicted tokens.

    Returns:
        list[tuple[str, str, int]]: (gt_token, pred_token, distance)
    """
    
    gt_tokens = [str(token).lower() for token in gt_tokens]
    pred_tokens = [str(token).lower() for token in pred_tokens]

    n_gt = len(gt_tokens)
    n_pred = len(pred_tokens)

    if n_gt == 0:
        return []
    if n_pred == 0:
        return [(gt, "", len(gt)) for gt in gt_tokens]

    if (n_pred > n_gt):
        pad = max(0, n_pred - n_gt)
        gt_aug = gt_tokens + [""] * pad
        pred_aug = pred_tokens
    elif (n_gt > n_pred):
        pad = max(0, n_gt - n_pred)
        pred_aug = pred_tokens + [""] * pad
        gt_aug = gt_tokens
    else:
        gt_aug = gt_tokens
        pred_aug = pred_tokens

    cost = np.zeros((len(gt_aug), len(pred_aug)), dtype=np.float32)
    for i, gt in enumerate(gt_aug):
        for j, pred in enumerate(pred_aug):
            if gt == "":
                cost[i, j] = len(pred)
            elif pred == "":
                cost[i, j] = len(gt)
            else:
                cost[i, j] = lev.distance(gt, pred)

    row_ind, col_ind = linear_sum_assignment(cost)

    mapping = []
    # We only keep the pairs that correspond to the original GT tokens (not the padded ones)
    for i, j in zip(row_ind, col_ind):
        if i < n_gt:
            mapping.append((gt_tokens[i], pred_aug[j], int(cost[i, j])))

    return mapping

def dice_chars(gt_token: str, pred_token: str) -> float:
    """
    Compute the Dice coefficient between two strings based on character bigrams.
    Returns a float between 0 and 1, where 1 means the strings are identical.
    """
    gt_bigrams = set([gt_token[i:i+2] for i in range(len(gt_token)-1)])
    pred_bigrams = set([pred_token[i:i+2] for i in range(len(pred_token)-1)])
    
    intersection = len(gt_bigrams.intersection(pred_bigrams))
    total_bigrams = len(gt_bigrams) + len(pred_bigrams)
    
    if total_bigrams == 0:
        return 1.0 if gt_token == pred_token else 0.0
    
    dice_index = (2 * intersection) / total_bigrams
    return dice_index

def compute_metrics(gt_token: str, pred_token: str, confidence: float = 0.0) -> tuple:
    """
    Compute metrics based on GT and predicted tokens.
    Returns: levenshtein distance, levenshtein ratio, character error rate (CER), and exact match boolean.
    """
    gt_token = str(gt_token).lower()
    pred_token = str(pred_token).lower()
    
    lev_dist = min(lev.distance(gt_token, pred_token), len(gt_token)) if len(gt_token) > 0 else len(pred_token)
    lev_ratio = lev_dist / max(len(gt_token), 1) if len(gt_token) > 0 else 0
    cer = lev_dist / max(len(pred_token), 1) if len(pred_token) > 0 else 0
    exact_match = gt_token == pred_token
    dice_index = dice_chars(gt_token, pred_token)
    
    # Round the metrics to 4 decimal places for consistency
    lev_dist = round(lev_dist, 4)
    lev_ratio = round(lev_ratio, 4)
    cer = round(cer, 4)
    dice_index = round(dice_index, 4)

    return lev_dist, lev_ratio, cer, exact_match, dice_index

def compute_dataset_metrics(file_path:str):
    """
    Compute metrics for the entire dataset.
    Returns a dictionary with metrics for each image and average metrics for the dataset.
    """
    
    results = {}
    cer_sum = 0
    lev_dist_sum = 0
    lev_ratio_sum = 0
    exact_match_sum = 0
    dice_sum = 0
    # Open the predictions JSON file
    with open(file_path, 'r', encoding="utf-8") as f:
        #print(f"Evaluating predictions for {file_path}")
        predictions = json.load(f)
        
        # For each image, we will compute the metrics and store them in a results dictionary
        for img_name, pred_data in predictions.items():
            if img_name == "batch_info":
                continue
            # Getting the ground truth tokens for the current image
            gt_tokens = gt_words.get(img_name, [])
            # Getting the predicted tokens for the current image
            pred_tokens, confidences = [word for word, conf in pred_data.get("predictions", [])], [conf for word, conf in pred_data.get("predictions", [])]
            # Get the processing time for the current image
            process_time = pred_data.get("time", 0)
            # Aligning the GT and predicted tokens using Hungarian matching
            aligned_pairs = hungarian_align(gt_tokens, pred_tokens)
            img_results = []
            
            # Compute metrics for each aligned pair
            for gt_token, pred_token, _ in aligned_pairs:
                lev_dist, lev_ratio, cer, exact_match, dice = compute_metrics(gt_token, pred_token)
                cer_sum += cer
                dice_sum += dice
                
                lev_dist_sum += lev_dist
                lev_ratio_sum += lev_ratio
                exact_match_sum += int(exact_match)
                metrics = {
                    "ground_truth": gt_token,
                    "predicted": pred_token,
                    "confidence": confidences[pred_tokens.index(pred_token)] if pred_token in pred_tokens else None,
                    "lev_dist": lev_dist,
                    "lev_ratio": lev_ratio,
                    "cer": cer,
                    "dice_index": dice,
                    "exact_match": exact_match
                }
                img_results.append(metrics)
            img_results.append({"processing_time": process_time})

            results[img_name] = img_results
            
    # Calculate average metrics for the dataset
    num_images = len(gt_words)
    avg_cer = round(cer_sum / num_images, 4) if num_images > 0 else 0
    avg_leven_dist = round(lev_dist_sum / num_images, 4) if num_images > 0 else 0
    avg_leven_ratio = round(lev_ratio_sum / num_images, 4) if num_images > 0 else 0
    avg_exact_match = round(exact_match_sum / num_images, 4) if num_images > 0 else 0
    avg_dice_index = round(dice_sum / num_images, 4) if num_images > 0 else 0

    results["average_metrics"] = {
        "avg_cer": avg_cer,
        "avg_levenshtein_distance": avg_leven_dist,
        "avg_levenshtein_ratio": avg_leven_ratio,
        "avg_exact_match": avg_exact_match,
        "avg_dice_index": avg_dice_index
    }
    
    return results

### 4.1.2 Evaluation
This step create the files in `/output/results` folder, which contains the evaluation of the models on the dataset.

In [ ]:
datasets_to_evaluate = ["ICDAR03/apanar",
                        "ICDAR03/lfsosa",
                        "ICDAR03/ryoungt1",
                        "ICDAR03/ryoungt2",
                        "ICDAR13/train",
                        "ICDAR13/test",
                        "kahua-ml/flattened-nameplate",
                        "SVT/train",
                        "icare/train"
                        ]

for dataset in datasets_to_evaluate:
    main, sub = dataset.split("/")
    if not os.path.exists(f"{RESULTS_PATH}{main}/{sub}"):
        print(f"Results directory for {dataset} does not exist. Skipping.")
        continue
    print(f"Evaluating dataset: {dataset}")
    gt_words = get_words_lists(f"{main}/{sub}")
    for files in os.listdir(f"{SAVE_PATH}{main}"):
        if files.startswith(sub) and files.endswith(".json"):
            file_path = os.path.join(f"{SAVE_PATH}{main}", files)
            with open(file_path, "r", encoding="utf-8") as f:
                results = compute_dataset_metrics(file_path)
            
            model_name = files.removesuffix(".json")
            model_name = "_".join(model_name.split("_")[1:])  # Extract model name from file name
            result_path = os.path.join(f"{RESULTS_PATH}{main}/{sub}", f"{model_name}.json")
            os.makedirs(os.path.dirname(result_path), exist_ok=True)
            with open(result_path, 'w', encoding="utf-8") as f:
                json.dump(results, f, indent=2, ensure_ascii=False)


## 4.2 Analysis

### 4.2.1 Definitions

In [ ]:
def build_datagram(results_dir_path: str, dataset_excludes: list[str] = [], subset_excludes: list[str] = []) -> tuple[pd.DataFrame, pd.DataFrame]:
    data_rows = []
    time_rows = []

    for dataset in os.listdir(results_dir_path):
        dataset_path = os.path.join(results_dir_path, dataset)
        if not os.path.isdir(dataset_path):
            continue
        if dataset in dataset_excludes:
            continue
        for subset in os.listdir(dataset_path):
            if subset in subset_excludes:
                continue
            subset_path = os.path.join(dataset_path, subset)
            if not os.path.isdir(subset_path):
                continue
            for file in os.listdir(subset_path):
                file_path = os.path.join(subset_path, file)
                if not file.endswith(".json"):
                    continue
                model_name = file.replace(".json", "")
                with open(file_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                for image_name, words in data.items():
                    if image_name == "average_metrics":
                        continue
                    
                    for metrics in words[:-1]:
                        data_rows.append({
                            "dataset": dataset,
                            "subset": subset,
                            "model": model_name,
                            "image_name": image_name,
                            "gt_word": metrics.get("ground_truth", ""),
                            "pred_word": metrics.get("predicted", ""),
                            "confidence": metrics.get("confidence", np.nan),
                            "lev_dist": metrics.get("lev_dist", np.nan),
                            "lev_ratio": metrics.get("lev_ratio", np.nan),
                            "cer": metrics.get("cer", np.nan),
                            "exact_match": metrics.get("exact_match", np.nan)
                        })
                    time_rows.append({
                        "dataset": dataset,
                        "subset": subset,
                        "model": model_name,
                        "image_name": image_name,
                        "time": words[-1].get("processing_time", np.nan)
                    })

    return pd.DataFrame(data_rows), pd.DataFrame(time_rows)


In [ ]:
metric_df, time_df = build_datagram("output/results")

### 4.2.2 Analysis

Different types of heuristics can be used to evaluate the performance of the models. The following heuristics are available:

In [ ]:
def h_exact_match(gt: str, pred: str) -> float:
    """Use the exact match metric to evaluate the prediction. Returns 1.0 if the prediction is correct, 0.0 otherwise."""
    return 1.0 if gt == pred else 0.0

def h_negative_lev(gt: str, pred: str) -> float:
    """Use the Levenshtein distance metric to evaluate the prediction.
    Returns an int representing the number of correct characters in the prediction, where 0 means completely wrong and len(gt) means perfect match."""
    # "Start" and "ptart" gets a score of 4 because levenshtein distance is 1 and the length of the gt is 5, so 5 - 1 = 4 correct characters.
    score = len(gt) - lev.distance(gt, pred)
    return max(score, 0)

def h_cer(gt: str, pred: str) -> float:
    """Use the character error rate (CER) metric to evaluate the prediction.
    Returns a float between 0.0 and 1.0,
    where 0.0 means perfect match and 1.0 means completely wrong."""
    if len(gt) == 0:
        return 1.0
    return lev.distance(gt, pred) / len(gt)

def h_prefix(gt: str, pred: str) -> float:
    """Use the prefix match metric to evaluate the prediction.
    Returns a float between 0.0 and 1.0, 
    where 1.0 means perfect match and 0.0 means completely wrong."""
    score = 0
    for g, p in zip(gt, pred):
        if g == p:
            score += 1
        else:
            break
    return score / max(len(gt), 1)

def h_soft_accuracy(gt: str, pred: str, threshold: float = 0.75) -> float:
    """Use the soft accuracy metric to evaluate the prediction.
    Returns 1.0 if the CER is below the threshold, 0.0 otherwise."""
    return 1.0 if h_cer(gt, pred) <= threshold else 0.0

def h_semantic(gt: str, pred: str) -> float:
    """Use the semantic similarity metric to evaluate the prediction.
    Returns a float between 0.0 and 1.0 which represents the similarity between the two strings, where 1.0 means perfect match and 0.0 means completely wrong."""
    return sm(None, gt, pred).ratio()

def h_dice(gt: str, pred: str) -> float:
    """Use the Dice coefficient metric to evaluate the prediction.
    Returns a float between 0.0 and len(gt) which represents the similarity between the two strings, where len(gt) means perfect match and 0.0 means completely wrong."""
    score = dice_chars(gt, pred)
    weighted_score = score * len(gt)
    return weighted_score

Fonctions to plot the results of the evaluation

In [ ]:
def get_filtered_dataframe(dataframe, dataset_name, model_names):
    """
    Filter the dataframe based on the dataset name and model names.
    Args:
        dataframe (pd.DataFrame): DataFrame containing the metrics.
        dataset_name (str): Name of the dataset in the format "main/subset".
        model_names (list): List of model names to filter.
    Returns:
        pd.DataFrame: Filtered DataFrame.
    """
    if "/" in dataset_name:
        main, sub = dataset_name.split("/")
        filtered_dataframe = dataframe[(dataframe['dataset'] == main) & (dataframe['subset'] == sub)]
    else:
        main, sub = dataset_name, None
        filtered_dataframe = dataframe[(dataframe['dataset'] == main)]
        
    filtered_dataframe = filtered_dataframe[filtered_dataframe['model'].isin(model_names)]
    
    return filtered_dataframe

def plot_edit_distance_distribution(dataframe, dataset_name, model_names, distance_limit:int = 12, title="Distribution des distances d'édition"):
    """
    Plot the distribution of Levenshtein distances for a given dataset and model(s).
    Args:
        dataframe (pd.DataFrame): DataFrame containing the metrics.
        dataset_name (str): Name of the dataset in the format "main/subset".
        model_names (list): List of model names to plot.
        title (str): Title of the plot.
    """
    data = {}
    found_models = []
    filtered_dataframe = get_filtered_dataframe(dataframe, dataset_name, model_names)

    max_distance = filtered_dataframe['lev_dist'].max()
    plot_distance_limit = min(max_distance, distance_limit)
    print(f"Max distance for dataset {dataset_name}: {max_distance} for word: {filtered_dataframe.loc[filtered_dataframe['lev_dist'].idxmax()]['gt_word']} but we will limit the plot to {distance_limit}.")

    for model in model_names:
        if model not in filtered_dataframe['model'].unique():
            print(f"Model {model} not found in the dataframe. Skipping.")
            continue
        model_data = filtered_dataframe[(filtered_dataframe['model'] == model) & (filtered_dataframe['lev_dist'] <= plot_distance_limit)]
        nb_words = len(model_data)
        distances = model_data['lev_dist'].dropna().astype(int)
        data[model] = distances
        found_models.append(model)
        
    plt.figure(figsize=(10, 6))
    colors = sns.color_palette("Set1", len(data))
    ax = plt.gca()

    all_distances = np.concatenate([d.to_numpy() for d in data.values() if len(d) > 0])
    if all_distances.size == 0:
        return

    x = np.arange(int(all_distances.min()), int(all_distances.max()) + 1)
    bar_width = 0.8 / max(len(data), 1)

    for idx, (model, distances) in enumerate(data.items()):
        counts = distances.value_counts().reindex(x, fill_value=0).to_numpy()
        offset = (idx - (len(data) - 1) / 2) * bar_width
        ax.bar(x + offset, counts, width=bar_width, alpha=0.6, color=colors[idx], label=model, align="center")
         
         
    x_ticks = np.arange(plot_distance_limit + 1)
    ax.set_xticks(x_ticks)
    ax.tick_params(axis='x', which='minor', bottom=False)
    ax.yaxis.set_minor_locator(tck.AutoMinorLocator())
    ax.set_xlabel("Distance d'édition (Levenshtein)", fontsize=12)
    ax.set_ylabel("Nombre de mots", fontsize=12)
    ax.set_title(title, fontsize=14)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    acceptance_threshold = ACCEPT_RATE * nb_words
    plt.ylim(0, nb_words * 1.1 )
    satisfaction_rate = SATISFACTION_RATE * nb_words
    ax.axhline(y=satisfaction_rate, color='green', linestyle='--')
    ax.axhline(y=acceptance_threshold, color='red', linestyle='--')
    
    # We add the two horizontal lines for acceptance and satisfaction rates, but we need to add them to the legend manually
    handles, labels = ax.get_legend_handles_labels()
    handles.append(plt.Line2D([0], [0], color='red', linestyle='--'))
    labels.append(f"Acceptation ({ACCEPT_RATE*100}%)")
    handles.append(plt.Line2D([0], [0], color='green', linestyle='--'))
    labels.append(f"Satisfaction ({SATISFACTION_RATE*100}%)")
    plt.legend(handles, labels, title="Modèles", loc="upper right", framealpha=1.0)
    plt.tight_layout()
    plt.show()

def plot_edit_distance_inv_cumulative(dataframe, dataset_name, model_names, distance_limit:int = 12, title="Distribution inverse des distances d'édition"):
    data = {}
    found_models = []

    # Filtrage dataset
    if "/" in dataset_name:
        main, sub = dataset_name.split("/")
        filtered_dataframe = dataframe[(dataframe['dataset'] == main) & (dataframe['subset'] == sub)]
    else:
        main, sub = dataset_name, None
        filtered_dataframe = dataframe[(dataframe['dataset'] == main)]

    max_distance = filtered_dataframe['lev_dist'].max()
    plot_distance_limit = min(max_distance, distance_limit)

    print(f"Max distance for dataset {dataset_name}: {max_distance} (limitée à {plot_distance_limit}).")

    # Collecte des distances par modèle
    for model in model_names:
        if model not in filtered_dataframe['model'].unique():
            print(f"Model {model} not found in the dataframe. Skipping.")
            continue

        model_data = filtered_dataframe[
            (filtered_dataframe['model'] == model) &
            (filtered_dataframe['lev_dist'] <= plot_distance_limit)
        ]

        distances = model_data['lev_dist'].dropna().astype(int)
        data[model] = distances
        found_models.append(model)

    # Plot
    plt.figure(figsize=(10, 6))
    ax = plt.gca()
    colors = sns.color_palette("Set1", len(data))

    # Axe X = distances possibles
    x = np.arange(0, plot_distance_limit + 1)

    # Courbe cumulative inversée
    for idx, (model, distances) in enumerate(data.items()):
        # Comptage par distance
        counts = distances.value_counts().reindex(x, fill_value=0).to_numpy()

        # Cumul inversé : nombre de cas ayant une distance <= x
        cumulative_counts = np.cumsum(counts)

        ax.plot(
            x,
            cumulative_counts,
            color=colors[idx],
            label=model,
            linewidth=2
        )

    # Mise en forme
    ax.set_title(title)
    ax.yaxis.set_minor_locator(tck.AutoMinorLocator())
    ax.yaxis.set_minor_formatter(tck.NullFormatter())
    ax.set_xticks(x)
    ax.set_xlabel("Distance d'édition (Levenshtein)", fontsize=12)
    ax.set_ylabel("Nombre de cas ayant au plus cette distance", fontsize=12)
    ax.set_title(title, fontsize=14)
    plt.grid(axis='both', linestyle='--', alpha=0.8)
    plt.grid(axis='y', which='minor', linestyle=':', alpha=0.6)
    plt.legend(found_models, title="Modèles", loc="lower right")
    plt.tight_layout()
    plt.show()

def plot_confidence_distribution(dataframe, dataset_name, model_names, title="Distribution des scores de confiance"):
    """
    Plot the distribution of confidence scores for a given dataset and model(s).
    Args:
        dataframe (pd.DataFrame): DataFrame containing the metrics.
        dataset_name (str): Name of the dataset in the format "main/subset".
        model_names (list): List of model names to plot.
        title (str): Title of the plot.
    """
    data = {}
    found_models = []
    
    if "/" in dataset_name:
        main, sub = dataset_name.split("/")
        filtered_dataframe = dataframe[(dataframe['dataset'] == main) & (dataframe['subset'] == sub)]
    else:
        filtered_dataframe = dataframe[(dataframe['dataset'] == dataset_name)]

    for model in model_names:
        if model not in filtered_dataframe['model'].unique():
            print(f"Model {model} not found in the dataframe. Skipping.")
            continue
        model_data = filtered_dataframe[filtered_dataframe['model'] == model]
        confidences = model_data['confidence'].replace(np.nan, 0).astype(float)
        data[model] = confidences
        found_models.append(model)
        print(f"Model {model} has {len(confidences)} confidence scores for dataset {dataset_name}.")
        print(f"Max confidence: {confidences.max()}, Min confidence: {confidences.min()}")
        
    plt.figure(figsize=(10, 6))
    ax = plt.gca()
    colors = sns.color_palette("Set1", len(data))

    all_confidences = np.concatenate([d.to_numpy() for d in data.values() if len(d) > 0])
    if all_confidences.size == 0:
        return

    x = np.linspace(0, 1, 100)
    bar_width = 0.8 / max(len(data), 1)

    for idx, (model, confidences) in enumerate(data.items()):
        counts, _ = np.histogram(confidences, bins=x)
        offset = (idx - (len(data) - 1) / 2) * bar_width
        ax.bar(
            x[:-1] + offset,
            counts,
            width=bar_width,
            alpha=0.6,
            color=colors[idx],
            label=model,
            align="edge",
        )
         
    ax.set_xlabel("Score de confiance")
    ax.set_ylabel("Nombre de mots")
    ax.set_title(title)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend(found_models, title="Modèles", loc="upper right")
    plt.tight_layout()
    plt.show()

def plot_exact_match_distribution_inv(dataframe, dataset_name, model_names, title="Distribution des correspondances exactes"):
    """
    Plot the inverse distribution of exact match scores for a given dataset and model(s). X-axis represents the number of words, Y-axis represents the number of words that were correctly predicted (exact match)., 
    Args:
        dataframe (pd.DataFrame): DataFrame containing the metrics.
        dataset_name (str): Name of the dataset in the format "main/subset".
        model_names (list): List of model names to plot.
        title (str): Title of the plot.
    """
    colors = sns.color_palette("Set1", len(model_names))
    data = {}
    found_models = []
    if "/" in dataset_name:
        main, sub = dataset_name.split("/")
        filtered_dataframe = dataframe[(dataframe['dataset'] == main) & (dataframe['subset'] == sub)]
    else:
        filtered_dataframe = dataframe[(dataframe['dataset'] == dataset_name)]

    for model in model_names:
        if model not in filtered_dataframe['model'].unique():
            print(f"Model {model} not found in the dataframe. Skipping.")
            continue
        model_data = filtered_dataframe[filtered_dataframe['model'] == model]
        exact_matches = model_data['exact_match'].replace(np.nan, 0).astype(int)
        data[model] = exact_matches
        found_models.append(model)
        
    plt.figure(figsize=(10, 6))
    ax = plt.gca()

    all_exact_matches = np.concatenate([d.to_numpy() for d in data.values() if len(d) > 0])
    if all_exact_matches.size == 0:
        return

    y = np.arange(0, 2)  # Exact match can be 0 or 1
    bar_height = 0.8 / max(len(data), 1)
    
    for idx, (model, exact_matches) in enumerate(data.items()):
        counts = exact_matches.value_counts().reindex(y, fill_value=0).to_numpy()
        offset = (idx - (len(data) - 1) / 2) * bar_height
        ax.barh(
            y + offset,
            counts,
            height=bar_height,
            alpha=0.6,
            color=colors[idx],
            label=model,
            align="center",
        )
    ax.set_yticks(y)
    ax.set_yticklabels(["Incorrect", "Correct"])
    ax.set_xlabel("Nombre de mots")
    ax.set_title(title)
    ax.xaxis.set_minor_locator(tck.AutoMinorLocator())
    ax.xaxis.set_minor_formatter(tck.NullFormatter())
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.legend(found_models, title="Modèles", loc="lower right")
    plt.tight_layout()
    plt.show()

def plot_cactus_chart_cer(dataframe, dataset_name, model_names, title="Cactus Plot"):
    """
    Plot a cactus chart for the given dataset and model(s) using CER values as scores.
    Args:
        dataframe (pd.DataFrame): DataFrame containing the metrics.
        dataset_name (str): Name of the dataset in the format "main/subset".
        model_names (list): List of model names to plot.
        title (str): Title of the plot.
    """
    plt.figure(figsize=(10, 6))
    ax = plt.gca()
    colors = sns.color_palette("Set1", len(model_names))
    
    if "/" in dataset_name:
        main, sub = dataset_name.split("/")
        filtered_dataframe = dataframe[(dataframe['dataset'] == main) & (dataframe['subset'] == sub)]
    else:
        filtered_dataframe = dataframe[(dataframe['dataset'] == dataset_name)]

    for idx, model in enumerate(model_names):
        if model not in filtered_dataframe['model'].unique():
            print(f"Model {model} not found in the dataframe. Skipping.")
            continue
        model_data = filtered_dataframe[filtered_dataframe['model'] == model]
        sorted_cer = np.sort(model_data['cer'].dropna())
        cumulative = np.arange(1, len(sorted_cer) + 1)
        ax.plot(sorted_cer, cumulative, label=model, color=colors[idx])

    ax.set_ylabel("Nombre de mots")
    ax.set_title(title)
    plt.grid(axis='both', linestyle='--', alpha=0.7)
    plt.legend(title="Modèles", loc="lower right")
    plt.tight_layout()
    plt.show()

def plot_cactus_chart_time(dataframe, dataset_name, model_names, time_limit: int, title="Cactus Plot"):
    """
    Plot a cactus chart for the given dataset and model(s) using processing time as scores.
    Args:
        dataframe (pd.DataFrame): DataFrame containing the metrics.
        dataset_name (str): Name of the dataset in the format "main/subset".
        model_names (list): List of model names to plot.
        time_limit (int): Maximum processing time to consider, in seconds.
        title (str): Title of the plot.
    """
    plt.figure(figsize=(10, 6))
    ax = plt.gca()
    colors = sns.color_palette("Set1", len(model_names))
    number_of_words = 0
    
    if "/" in dataset_name:
        main, sub = dataset_name.split("/")
        filtered_dataframe = dataframe[(dataframe['dataset'] == main) & (dataframe['subset'] == sub)]
    else:
        filtered_dataframe = dataframe[(dataframe['dataset'] == dataset_name)]
    

    for idx, model in enumerate(model_names):
        if model not in filtered_dataframe['model'].unique():
            print(f"Model {model} not found in the dataframe. Skipping.")
            continue
        model_data = filtered_dataframe[filtered_dataframe['model'] == model]
        sorted_time = np.sort(model_data['time'])
        number_of_words = len(sorted_time)
        # After sorting the times, we compute the cumulative number of words that can be processed within the time limit.
        # The time limit is for the whole dataset, not for each word. So we need to compute the cumulative sum of the times and find the index where the cumulative sum exceeds the time limit.
        # The x-axis will be the acumulated time, and the y-axis will be the number of words processed within that time.
        cumulative_time = np.cumsum(sorted_time)
        within_limit = cumulative_time <= time_limit
        ax.plot(cumulative_time[within_limit], np.arange(1, np.sum(within_limit) + 1), label=model, color=colors[idx])

    ax.set_xlabel("Temps de traitement (secondes)", fontsize=12)
    ax.set_ylabel("Nombre d'images traitées", fontsize=12)
    ax.yaxis.set_major_locator(tck.MaxNLocator(integer=True))
    ax.set_title(title)
    ax.xaxis.set_minor_locator(tck.AutoMinorLocator())
    ax.xaxis.set_minor_formatter(tck.NullFormatter())
    ax.yaxis.set_minor_formatter(tck.NullFormatter())
    
    plt.grid(axis='both', which='major', linestyle='--', alpha=0.7)
    plt.grid(axis='both', which='minor', linestyle=':', alpha=0.5)
    ax.axhline(y=number_of_words, color='gray', linestyle='--')
    y_ticks = list(ax.get_yticks()) + [number_of_words]
    ax.set_yticks(y_ticks)
    plt.ylim(bottom=0, top= 1.1 * number_of_words)
    plt.legend(title="Modèles", loc="lower right")
    plt.tight_layout()
    plt.title(f"Evolution du nombre d'images traitées sur {dataset_name} en fonction du temps", fontsize=14)
    plt.show()

def plot_cactus_chart_h(dataframe: pd.DataFrame, dataset_name: str, model_names: list, h_name: str, title="Cactus Plot"):
    """
    Plot a cactus chart for the given dataset and model(s) using a specified heuristic as scores.
    Args:
        dataframe (pd.DataFrame): DataFrame containing the metrics.
        dataset_name (str): Name of the dataset in the format "main/subset".
        model_names (list): List of model names to plot.
        h_name (str): Name of the heuristic method to use to compute scores.
        title (str): Title of the plot.
    """
    plt.figure(figsize=(10, 6))
    ax = plt.gca()
    colors = sns.color_palette("Set1", len(model_names))
    
    if "/" in dataset_name:
        main, sub = dataset_name.split("/")
        filtered_dataframe = dataframe[(dataframe['dataset'] == main) & (dataframe['subset'] == sub)]
    else:
        filtered_dataframe = dataframe[(dataframe['dataset'] == dataset_name)]

    for idx, model in enumerate(model_names):
        if model not in filtered_dataframe['model'].unique():
            print(f"Model {model} not found in the dataframe. Skipping.")
            continue
        model_data = filtered_dataframe[filtered_dataframe['model'] == model]
        scores = model_data.apply(lambda row: globals()[f"{h_name}"](row['gt_word'], row['pred_word']), axis=1)
        sorted_scores = np.sort(scores.dropna())
        cumulative = np.arange(1, len(sorted_scores) + 1)
        ax.plot(sorted_scores, cumulative, label=model, color=colors[idx])
        
    ax.set_xlabel(f"Score ({h_name})")
    ax.set_ylabel("Nombre de mots")
    ax.xaxis.set_minor_locator(tck.AutoMinorLocator())
    ax.xaxis.set_minor_formatter(tck.NullFormatter())
    ax.set_title(title)
    plt.grid(axis='both', linestyle='--', alpha=0.7)
    plt.legend(title="Modèles", loc="upper left")
    plt.tight_layout()
    plt.show()
    
def plot_cactus_chart_score_over_time(metric_df: pd.DataFrame, time_df: pd.DataFrame, dataset_name: str, model_names: list, h_name: str = "h_negative_lev", title="Cactus Plot"):
    """
    Plot a cactus chart for the given dataset and model(s) using a specified heuristic as scores over time.
    Args:
        metric_df (pd.DataFrame): DataFrame containing the metrics.
        time_df (pd.DataFrame): DataFrame containing the processing times.
        dataset_name (str): Name of the dataset in the format "main/subset".
        model_names (list): List of model names to plot.
        h_name (str): Name of the heuristic method to use to compute scores.
        title (str): Title of the plot.
    """
    plt.figure(figsize=(10, 6))
    ax = plt.gca()
    colors = sns.color_palette("Set1", len(model_names))

    # --- Filter dataset ---
    if "/" in dataset_name:
        main, sub = dataset_name.split("/")
        metric_df = metric_df[(metric_df['dataset'] == main) & (metric_df['subset'] == sub)]
        time_df = time_df[(time_df['dataset'] == main) & (time_df['subset'] == sub)]
    else:
        metric_df = metric_df[(metric_df['dataset'] == dataset_name)]
        time_df = time_df[(time_df['dataset'] == dataset_name)]

    # --- Compute score per word (no collapse!) ---
    metric_df["score"] = metric_df.apply(
        lambda row: globals()[f"{h_name}"](row["gt_word"], row["pred_word"]),
        axis=1
    )

    # --- Title depending on heuristic ---
    match h_name:
        case "h_exact_match":
            title = f"Correspondance exacte au fil du temps pour {dataset_name}"
        case "h_negative_lev":
            title = f"Score de Levenshtein négatif au fil du temps pour {dataset_name}"
        case "h_cer":
            title = f"Taux d'erreur de caractères (CER) au fil du temps pour {dataset_name}"
        case "h_prefix":
            title = f"Correspondance de préfixe au fil du temps pour {dataset_name}"
        case "h_soft_accuracy":
            title = f"Précision douce au fil du temps pour {dataset_name}"
        case "h_semantic":
            title = f"Similarité sémantique au fil du temps pour {dataset_name}"
        case "h_dice":
            title = f"Coefficient de Dice au fil du temps pour {dataset_name}"
        case _:
            title = f"Score personnalisé ({h_name}) au fil du temps pour {dataset_name}"

    # --- Plot per model ---
    for idx, model in enumerate(model_names):

        m = metric_df[metric_df["model"] == model]
        t = time_df[time_df["model"] == model]

        if m.empty or t.empty:
            print(f"Model {model} not found. Skipping.")
            continue

        # Merge per-word metrics with per-image time
        merged = pd.merge(
            m,
            t[["image_name", "time"]],
            on="image_name",
            how="inner"
        )

        # Sort by time
        merged = merged.sort_values(by="time")

        # --- Cumulative score: per word ---
        cumulative_score = merged["score"].cumsum()

        # --- Cumulative time: per image (unique) ---
        unique_times = merged.drop_duplicates("image_name")["time"].cumsum()

        # Expand cumulative time to match number of words
        time_map = dict(zip(
            merged.drop_duplicates("image_name")["image_name"],
            unique_times
        ))
        cumulative_time = merged["image_name"].map(time_map)

        print(f"{model}: {len(merged)} words, {merged['image_name'].nunique()} images, total time = {unique_times.iloc[-1]:.2f}s")

        ax.plot(cumulative_score, cumulative_time, label=model, color=colors[idx])

    # --- Plot formatting ---
    ax.set_xlabel("Score cumulé (par mot)", fontsize=12)
    ax.set_ylabel("Temps cumulé (secondes, par image)", fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.xaxis.set_minor_locator(tck.AutoMinorLocator())
    ax.xaxis.set_minor_formatter(tck.NullFormatter())
    ax.yaxis.set_minor_locator(tck.AutoMinorLocator())
    ax.yaxis.set_minor_formatter(tck.NullFormatter())
    plt.grid(True, linestyle="--", alpha=0.8)
    plt.grid(axis='y', which='minor', linestyle=':', alpha=0.6)
    plt.legend(title="Models")
    plt.tight_layout()
    plt.show()

def plot_empty_predictions_distribution(dataframe, dataset_name, model_names, title="Images with Empty Predictions"):
    """
    Plot the number of images with empty predictions for a given dataset and model(s).
    Args:
        dataframe (pd.DataFrame): DataFrame containing the metrics.
        dataset_name (str): Name of the dataset in the format "main/subset".
        model_names (list): List of model names to plot.
        title (str): Title of the plot.
    """
    colors = sns.color_palette("Set1", len(model_names))
    data = {}
    found_models = []
    
    if "/" in dataset_name:
        main, sub = dataset_name.split("/")
        filtered_dataframe = dataframe[(dataframe['dataset'] == main) & (dataframe['subset'] == sub)]
    else:
        filtered_dataframe = dataframe[(dataframe['dataset'] == dataset_name)]

    for model in model_names:
        if model not in filtered_dataframe['model'].unique():
            print(f"Model {model} not found in the dataframe. Skipping.")
            continue
        model_data = filtered_dataframe[filtered_dataframe['model'] == model]
        empty_predictions_count = model_data[model_data['pred_word'] == ""].shape[0]
        data[model] = empty_predictions_count
        found_models.append(model)
        
    plt.figure(figsize=(10, 6))
    ax = plt.gca()
    
    x = np.arange(len(data))
    counts = list(data.values())
    
    ax.bar(x, counts, color=colors[:len(data)], alpha=0.7)
    
    ax.set_xticks(x)
    ax.set_xticklabels(found_models)
    ax.set_xlabel("Modèles")
    ax.set_ylabel("Nombre de prédictions vides")
    ax.set_title(title)
    
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    


### 4.2.3 Plotting

In [ ]:
ACCEPT_RATE = 0.75
SATISFACTION_RATE = 0.90

In [ ]:
# List of models available for plotting:
# "easyocr", "PaddleOCR-VL-1.6", "LightOnOCR-2-1B", "gemma-3-4b-it", "GLM-OCR", "Qwen2.5-VL-7B-Instruct"
# List of heuristics available for plotting:
# "h_exact_match", "h_negative_lev", "h_cer", "h_prefix", "h_soft_accuracy", "h_semantic"
# List of datasets available for plotting:
# "ICDAR03/apanar", "ICDAR03/lfsosa", "ICDAR03/ryoungt1", "ICDAR03/ryoungt2", "ICDAR13/train", "ICDAR13/test", "kahua-ml/flattened-nameplate", "kahua-ml/nameplates-v2", "SVT/train", "icare/train"

dataset_name = "kahua-ml"  # Change this to the desired dataset name
all_m = ["easyocr", "easyocr_bb", "pytesseract", "pytesseract_bb", "PaddleOCR-VL-1.6", "GLM-OCR", "LightOnOCR-2-1B", "gemma-3-4b-it", "Qwen2.5-VL-7B-Instruct"]
ocr_m = ["easyocr", "easyocr_bb", "pytesseract", "pytesseract_bb"]
llm_m = ["PaddleOCR-VL-1.6", "GLM-OCR", "LightOnOCR-2-1B", "gemma-3-4b-it", "Qwen2.5-VL-7B-Instruct"]
model_name = set(ocr_m) # Change this to the desired model name(s)
list_of_heuristics = ["h_exact_match", "h_negative_lev", "h_cer", "h_prefix", "h_soft_accuracy", "h_semantic"]

plot_edit_distance_distribution(metric_df, dataset_name, model_name, distance_limit=8, title=f"Distribution des distances d'édition sur le dataset {dataset_name}")
plot_edit_distance_inv_cumulative(metric_df, dataset_name, model_name, title=f"Distribution inverse des distances d'édition sur le dataset {dataset_name}")
#plot_empty_predictions_distribution(metric_df, dataset_name, model_name, title=f"Images avec des prédictions vides sur le dataset {dataset_name}")
plot_exact_match_distribution_inv(metric_df, dataset_name, model_name, title=f"Distribution des correspondances exactes sur le dataset {dataset_name}")
plot_cactus_chart_time(time_df, dataset_name, model_name, time_limit=600, title=f"Evolution du nombre d'images traitées sur {dataset_name} en fonction du temps (max. 30s)")
#plot_cactus_chart_h(metric_df, dataset_name, model_name, h_name="h_dice", title=f"Cactus Plot sur {dataset_name} avec la métrique Levenshtein")
plot_cactus_chart_score_over_time(metric_df, time_df, dataset_name, model_name, h_name="h_negative_lev", title=f"Cactus Plot sur {dataset_name}")

In [ ]:
# We collect some statistics about the datasets (number of words, number of images, average number of words per image etc.)
dataset_stats = {}

for root, dirs, files in os.walk("./resources/datasets"):
    for file in files:
        if file.endswith("data.json"):
            file_path = os.path.join(root, file)
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            dic = {}
            sub = os.path.basename(root)
            main = os.path.basename(os.path.dirname(root))
            
            if main not in dataset_stats:
                dataset_stats[main] = {
                    "num_images": 0,
                    "num_words": 0,
                    "avg_words_per_image": 0.0,
                    "avg_word_length": 0.0
                }
            else:
                dic = dataset_stats[main]
                
            num_images = len(data)
            num_words = sum(len(words) for words in data.values())
            avg_words_per_image = round(num_words / num_images if num_images > 0 else 0, 2)
            avg_word_length = round(sum(len(word) for words in data.values() for word in words) / num_words if num_words > 0 else 0, 2)
            
            dic["num_images"] += num_images
            dic["num_words"] += num_words
            dic["avg_words_per_image"] = round(dic["num_words"] / dic["num_images"] if dic["num_images"] > 0 else 0, 2)
            dic["avg_word_length"] = round(sum(len(word) for words in data.values() for word in words) / dic["num_words"] if dic["num_words"] > 0 else 0, 2)
            

dataset_stats_df = pd.DataFrame.from_dict(dataset_stats, orient="index")
dataset_stats_df.head(15)

### 4.2.4 Environmental information & analysis

In [ ]:
environment_df = pd.read_csv("resources/emissions/emissions.csv")
environment_df = environment_df.drop(columns=["run_id",
                                              "experiment_id",
                                              "ram_power",
                                              "water_consumed",
                                              "country_name",
                                              "country_iso_code",
                                              "region",
                                              "cloud_provider", "cloud_region",
                                              "os", "python_version", "codecarbon_version",
                                              "longitude", "latitude", "ram_total_size", "tracking_mode", "on_cloud", "pue","wue"
                                              ])

In [ ]:
# For each row of the df, the value in the column "project_name" contains mainSet/subSet_modelName. And we need to split it into three columns: main, sub, model. We can do this by using the str.split() method and then assign the result to new columns.
# So it's first a / then a _ so we can use str.split() twice, first with / and then with _.

environment_df[["dataset", "model"]] = environment_df["project_name"].str.split("_", n=1, expand=True)
environment_df[["dataset", "subset"]] = environment_df["dataset"].str.split("/", n=1, expand=True)
environment_df = environment_df.drop(columns=["project_name"])

### 4.2.5 Pareto analysis